# Day05 下午个人项目：电商用户多维分析

**姓名：** 请填写  
**专题方向：** A / B / C / D / E

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# TODO：填写个人信息与专题
# =========================
STUDENT_NAME = "李晓丰"
TOPIC = "B"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

姓名： 李晓丰
专题： B
输入数据： c:\Users\Li Xiaofeng\Documents\Codex\2026-07-15\github-plugin-github-openai-curated-remote\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： c:\Users\Li Xiaofeng\Documents\Codex\2026-07-15\github-plugin-github-openai-curated-remote\output\day05_analysis


In [2]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 李晓丰
专题： B


### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [3]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-6个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,7-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,7-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,新用户,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,新用户,1



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,object
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,object
Gender,object
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [4]:
# 补充检查所需字段（保持模板要求）
for col, default in {
    "OrderCount": 0,
    "CouponUsed": 0,
    "CashbackAmount": 0,
    "DaySinceLastOrder": 0
}.items():
    if col not in df.columns:
        df[col] = default

if "TenureGroup" not in df.columns:
    if "Tenure" in df.columns:
        df["TenureGroup"] = pd.cut(
            df["Tenure"],
            bins=[-1, 12, 24, 48, 1000],
            labels=["0-12个月", "13-24个月", "25-48个月", "48个月以上"]
        )
    else:
        df["TenureGroup"] = "未知"

print("字段准备完成")

字段准备完成


In [5]:
# TODO 1：定义需要验收的核心字段
core_cols = [
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
]

# TODO 2：完成数据验收表
validation = pd.DataFrame({
    "检查项": ["行数", "列数", "CustomerID重复数", "核心字段缺失数", "Churn取值"],
    "结果": [
        df.shape[0],
        df.shape[1],
        df["CustomerID"].duplicated().sum(),
        df[core_cols].isna().sum().sum(),
        sorted(df["Churn"].unique().tolist())
    ]
})

display(validation)

,检查项,结果
0,行数,5630
1,列数,22
2,CustomerID重复数,0
3,核心字段缺失数,0
4,Churn取值,"[0, 1]"


In [6]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = {
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
}

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

检查点1通过


### 数据粒度说明

请用一句话说明一行数据代表什么：

> TODO：一行数据代表一位用户的一次完整用户记录，包含该用户的基本信息、消费行为、订单情况以及是否流失等信息。

请说明为什么`CustomerID`不能作为普通连续数值求平均：

> TODO：CustomerID是用户的唯一标识编号，只用于区分不同用户，不代表用户之间存在数值大小关系，因此不能进行求平均等连续数值运算。

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [7]:
# TODO：计算公共基础指标

overall_churn_rate = df["Churn"].mean()

overall_metrics = pd.DataFrame({
    "指标": [
        "用户数", "流失人数", "总体流失率", "平均订单数", "订单数中位数",
        "平均优惠券使用次数", "平均返现", "平均App使用时长",
        "平均满意度", "平均距上次下单天数"
    ],
    "数值": [
        len(df),
        df["Churn"].sum(),
        overall_churn_rate,
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean()
    ]
})
display(overall_metrics)


,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,总体流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券使用次数,1.72
6,平均返现,177.22
7,平均App使用时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [8]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

# TODO：将变量赋值为你计算的总体流失率
overall_churn_rate = df["Churn"].mean()

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")

检查点2通过


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> TODO：当前样本共有5630名用户，其中流失用户948名，总体流失率为16.84%；用户平均订单数为2.96，平均满意度为3.07，投诉用户数量为1604名，投诉率为28.49%。

## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [9]:
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}

print("当前专题：", TOPIC)

segment_field = "Complain"

segment_analysis = (
    df.groupby(segment_field)
      .agg(
          用户数=("CustomerID","count"),
          流失人数=("Churn","sum"),
          流失率=("Churn","mean"),
          平均满意度=("SatisfactionScore","mean")
      )
      .reset_index()
)
segment_analysis["平均订单数"] = df.groupby(segment_field)["OrderCount"].mean().values
display(segment_analysis)

当前专题： B


,Complain,用户数,流失人数,流失率,平均满意度,平均订单数
0,0,4026,440,0.11,3.09,3.00
1,1,1604,508,0.32,3.00,2.86


In [10]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

检查点3通过


### 单维专题分析记录

**本专题要回答的业务问题：**

> TODO：投诉、满意度与用户流失存在怎样的关联？

**数据现象：**

> TODO：根据投诉情况进行分组分析发现，未投诉用户共有4026名，其中流失用户440名，流失率为11%；投诉用户共有1604名，其中流失用户508名，流失率为32%。投诉用户的平均满意度为3.00，低于未投诉用户的3.09，平均订单数为2.86，也低于未投诉用户的3.00。

**可能解释：**

> TODO：投诉用户群体的流失率较高，可能与用户服务体验较差有关；投诉用户平均满意度较低，可能说明服务问题会影响用户留存。但该分析仅反映投诉与流失之间的相关关系，无法直接证明投诉一定导致用户流失，还需要结合投诉类型、处理结果等因素进一步验证。

## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [11]:
allowed_cross_fields = {
    "TenureGroup","Complain","PreferedOrderCat",
    "CityTier","PreferredLoginDevice","PreferredPaymentMode"
}

dim_1 = "Complain"
dim_2 = "TenureGroup"

cross_analysis = (
    df.groupby([dim_1, dim_2])
      .agg(
          用户数=("CustomerID","count"),
          流失人数=("Churn","sum"),
          流失率=("Churn","mean"),
          平均满意度=("SatisfactionScore","mean")
      )
      .reset_index()
)

cross_analysis["样本提示"] = cross_analysis["用户数"].apply(
    lambda x: "小样本" if x < 30 else "可观察"
)

display(cross_analysis)

,Complain,TenureGroup,用户数,流失人数,流失率,平均满意度,样本提示
0,0,0-6个月,1177,189,0.16,3.11,可观察
1,0,13-24个月,1053,43,0.04,3.16,可观察
2,0,24个月以上,304,0,0.00,3.00,可观察
3,0,7-12个月,1178,75,0.06,2.98,可观察
4,0,新用户,314,133,0.42,3.32,可观察
5,1,0-6个月,465,236,0.51,3.03,可观察
6,1,13-24个月,414,52,0.13,2.91,可观察
7,1,24个月以上,125,0,0.00,3.19,可观察
8,1,7-12个月,406,81,0.20,3.02,可观察
9,1,新用户,194,139,0.72,2.94,可观察


In [12]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的维度组合：**

> TODO：投诉用户（Complain=1）中的新用户（新用户）组合。

**该组合的用户数、流失率和比较对象：**

> TODO：该组合共有194名用户，流失用户139名，流失率为72%。相比之下，未投诉新用户（Complain=0，新用户）共有314名用户，流失率为42%；投诉新用户的流失率更高。

**是否存在小样本风险：**

> TODO：不存在明显小样本风险。该组合用户数为194，大于30的观察阈值，因此属于可观察样本。

**为什么不能直接写成因果结论：**

> TODO：因为该分析展示的是投诉、用户生命周期与流失之间的相关关系，无法证明投诉一定导致用户流失，用户流失可能还受到其他因素影响，需要结合更多变量和进一步分析进行验证。

## 任务5：输出统计报表（必做）

In [13]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output\day05_analysis\overall_metrics.csv


已输出： output\day05_analysis\segment_analysis.csv
已输出： output\day05_analysis\cross_analysis.csv


In [14]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(2, 6)
通过：cross_analysis.csv，形状为(10, 7)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论1


在____用户中，____指标为____，与____相比____。对应证据表：____。

> TODO：在投诉用户中，流失率指标为32%，与未投诉用户的11%相比更高。对应证据表：单维专题分析表（Complain分组分析）。

### 结论2

> TODO：在投诉新用户群体中，流失率最高，为72%，共有194名用户，其中139名用户流失；相比未投诉新用户流失率42%，该组合流失情况更明显。对应证据表：双维交叉分析表（Complain × TenureGroup）。

### 结论3

> TODO：投诉用户的平均满意度为3.00，低于未投诉用户的3.09；同时投诉用户平均订单数为2.86，低于未投诉用户的3.00，说明不同投诉状态用户在服务体验指标上存在差异。对应证据表：单维专题分析表。

### 分析限制

至少写明一项当前数据不能支持的分析，或一项可能影响结论的限制。

> TODO：当前分析基于用户投诉状态、满意度和流失结果进行相关性分析，无法证明投诉一定导致用户流失。此外，数据未包含具体投诉类型、投诉处理结果、服务响应时间等信息，可能影响对用户流失原因的进一步判断。

### 运营建议与验证方式

提出一项与分析结果对应的建议，并说明还需要哪些数据或方法验证效果。

> TODO：建议重点关注投诉新用户群体，加强新用户阶段的服务体验管理，例如优化投诉处理流程和提升问题解决效率。后续可结合投诉类型、处理时长、客服响应记录等数据进行进一步分析，并通过用户回访或服务策略实验验证优化措施是否能够降低流失率。


## 拓展任务（选做）

In [15]:
# 拓展任务2：将双维分析整理为第6天绘图使用的长表
# 每一行表示一个“维度组合 × 指标”，便于后续按指标分面或筛选绘图。
plot_long_table = (
    cross_analysis
    .melt(
        id_vars=[dim_1, dim_2, "样本提示"],
        value_vars=["用户数", "流失人数", "流失率", "平均满意度"],
        var_name="指标",
        value_name="数值"
    )
    .sort_values([dim_1, dim_2, "指标"])
    .reset_index(drop=True)
)

plot_long_path = OUTPUT_DIR / "cross_analysis_long.csv"
plot_long_table.to_csv(plot_long_path, index=False, encoding="utf-8-sig")

display(plot_long_table.head(12))
print("长表形状：", plot_long_table.shape)
print("已输出：", plot_long_path.relative_to(ROOT))

# 拓展任务检查：每个双维组合应对应4个指标，且导出文件不含多余索引列。
assert len(plot_long_table) == len(cross_analysis) * 4
assert set(plot_long_table["指标"]) == {"用户数", "流失人数", "流失率", "平均满意度"}
assert plot_long_path.exists()
assert not any(str(col).startswith("Unnamed") for col in pd.read_csv(plot_long_path).columns)
print("拓展任务2检查通过")


,Complain,TenureGroup,样本提示,指标,数值
0,0,0-6个月,可观察,平均满意度,3.11
1,0,0-6个月,可观察,流失人数,189.00
2,0,0-6个月,可观察,流失率,0.16
3,0,0-6个月,可观察,用户数,"1,177.00"
4,0,13-24个月,可观察,平均满意度,3.16
5,0,13-24个月,可观察,流失人数,43.00
6,0,13-24个月,可观察,流失率,0.04
7,0,13-24个月,可观察,用户数,"1,053.00"
8,0,24个月以上,可观察,平均满意度,3.00
9,0,24个月以上,可观察,流失人数,0.00


长表形状： (40, 5)
已输出： output\day05_analysis\cross_analysis_long.csv
拓展任务2检查通过


## 最终检查：GitHub提交前验收

In [16]:
required_files = [
    ROOT / "notebooks" / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

本地提交文件检查通过
请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。


### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
2. 哪个检查点最能帮助你发现错误？
3. 哪条结论最容易被误解为因果关系？
4. 如果增加一个字段，你最希望增加什么？
5. 第6天准备把哪张统计表转化为图表？为什么？